In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "leeuwen2017conservatism")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "VanLeeuwen 2017_Copy if better EJC van Leeuwen.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="leeuwen2017conservatism"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns

In [3]:
df.rename(columns={"subject": "ape",
    "test.date": "date",
    "trained.token": "trained_token",
    "stooge.token": "stooge_token",
    "demonstrations.observed": "demonstrations_observed",
    "green.choice": "green_choice",
    "blue.choice": "blue_choice",
    "grey.choice": "grey_choice"}, inplace=True)

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [5]:
df['date']= pd.to_datetime(df['date'],format='%m/%d/%Y')
df['year']= df['date'].dt.year
df['month']= df['date'].dt.month
df['day']= df['date'].dt.day

In [6]:
# df.columns
df.rename(columns={"ape": "participant"}, inplace=True)

In [7]:
condition_rename= [['banana','test'],
                   ['carrot','control']]
for x,y in condition_rename:
    df['condition'].replace(x, y, inplace=True)

In [8]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

In [9]:
leeuwen2017conservatism_standardized=df[['study_id','year', 'month', 'day',  'participant','age_in_years','sex','species',  
        'session', 'trial', 'condition',  'order', 'trained_token', 
       'stooge_token', 'demonstrations_observed', 'green_choice',
       'blue_choice', 'grey_choice', 'trained', 'observed', 'random', 'choice']]


In [10]:
comp_out_path_stand = os.path.join(out_pathway, 'leeuwen2017conservatism_standardized.csv')
leeuwen2017conservatism_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =leeuwen2017conservatism_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
leeuwen2017conservatism_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'leeuwen2017conservatism_glossary.csv')
leeuwen2017conservatism_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
